# Load and analyse the data
In this notebook we will explore the different ways to load the data, visualize it, and make a preliminary analysis.

In [1]:
import sys
sys.path.append("/Users/samuel/Código/TFG")

In [2]:
import librosa as lib
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import src.io
import src.datasets.hls
import src.datasets.nih
from pathlib import Path
import soundfile as sf
import hashlib

## HLS-CMDS Data

In [ ]:
#Load annotations for HLS-CMDS
dataDir = "../data/raw/HLS-CMDS"
#Load annotations for HLS-CMDS
lung = pd.read_csv(f"{dataDir}/LS.csv")

#Load lung sounds into array where each element is one file
lungFiles = sorted((i for i in os.listdir(f"{dataDir}/LS") if i.endswith(".wav")))
lungSounds = np.empty(len(lungFiles), dtype=object)
lungSR = np.empty(len(lungFiles), dtype=object)
for n, i in enumerate(lungFiles):
    f, sr = lib.load(f"{dataDir}/LS/{i}", sr=None)
    lungSounds[n] = f
    lungSR[n] = sr
print(type(lungFiles))
print(type(lungSounds))
print(type(lungSR))

<class 'list'>
<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


In [ ]:
for n, f in enumerate(lungSounds):
        fig, ax = plt.subplots(ncols=4, figsize=(20,4))
        nffts = [256, 512, 1024, 2048]
        for i in range(len(nffts)): 
            #Transform the amplitude to dB so we have a logarithmic instead of linear plot
            spec = lib.amplitude_to_db(np.abs(lib.stft(f, n_fft=nffts[i])), ref=np.max)
            img = lib.display.specshow(spec, ax=ax[i], sr=lungSR[n], hop_length=nffts[i] // 4,
                                    x_axis="time", y_axis="linear")
            ax[i].set_title(f"{lungFiles[n]}, n_ffts={nffts[i]}, sr={lungSR[n]}")

### Con esta visualización se concluye que el valor óptimo para n_fft es 256, pues es aquel con el que la imagen queda más definida

## NIH Data

In [3]:
#Load annotations for HLS-CMDS
dataDirNIH = "../data/raw/NIH"
lungNIH, lungFilesNIH, lungSoundsNIH, lungSRNIH = src.io.load_dataset_xlsx(dataDirNIH, "Data annotation.xlsx", "Audio Files")

lungNIH

,Age,Gender,Location,Sound type,Diagnosis
0,70,M,P L L,I E W,Asthma
1,52,F,P L L,E W,Asthma
2,50,F,P L L,I E W,Asthma
3,72,F,P R L,I C,Heart Failure + Lung Fibrosis
4,71,M,P R L,I C E W,Heart Failure + COPD
...,...,...,...,...,...
107,63,M,P R L,E W,COPD
108,26,M,P L M,N,N
109,62,M,P L L,E W,COPD
110,51,M,P R L,E W,COPD


# Create inventory and annotations file(s)

## Inventory

In [3]:
inventory_l = []
raw_data = "../data/raw"
subtype_bit_depth: dict[str, pd.Int64Dtype] = {'PCM_16': 16, 'PCM_24': 24, 'PCM_32': 32, 'PCM_U8': 8, 'FLOAT': 32, 'DOUBLE': 64, 'ULAW': None, 'ALAW': None, 'IMA_ADPCM': None, 'MS_ADPCM': None, 
                     'GSM610': None, 'G721_32': None, 'NMS_ADPCM_16': None, 'NMS_ADPCM_24': None, 'NMS_ADPCM_32': None, 'MPEG_LAYER_III': None}
for file in Path(raw_data).rglob("*.wav"):
    dataset = file.relative_to(raw_data).parts[0]

    match dataset:
        case "HLS":
            annotations = src.datasets.hls.read_annotations()
        case "NIH":
            annotations = src.datasets.nih.read_annotations()

    info = sf.info(file)
    sr_f = info.samplerate
    duration = info.duration
    channels = info.channels
    fmt = info.format_info
    #extra_info = info.extra_info
    subtype = info.subtype
    bit_depth = subtype_bit_depth[subtype]
    is_float = subtype in ('FLOAT', 'DOUBLE')
    with file.open("rb") as f: 
        hash = hashlib.file_digest(f, "sha256").hexdigest()


    row = {"file": file.name, "dataset": dataset, "patient": None, "position": None, "hash": hash, "sample_rate": sr_f, "channels": channels, "bit_depth": bit_depth, "duration": duration, "annotation": None}

    inventory_l.append([
        file.name, dataset, None, None, hash, sr_f, channels, bit_depth, duration, None,
    ])

inventory = pd.DataFrame(inventory_l, columns=["file", "dataset", "patient", "position", "hash", "sample_rate", "channels", "bit_depth", "duration", "annotation"])
inventory.to_csv("../data/interim/inventory.csv")
    

FileNotFoundError: [Errno 2] No such file or directory: '/Users/samuel/Código/TFG/src/data/NIH/Data annotation.xlsx'